# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-khaled123/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding — "What Predicts Health?" (ML Appendix, Random Forest → health score).** Average Position
(43%) and Impressions (32%) are reported as the top predictors of the FlyRank Health Score. *My
question:* Health Score is explicitly defined earlier in the paper as `Impressions (30 pts) +
Position (30 pts) + CTR (20 pts) + Scroll depth (20 pts)` — i.e. Position and Impressions are
literally components the target is built from. A model finding that its own ingredients predict it
is closer to a definitional check than a discovery. The paper itself flags this ("importance is
descriptive rather than causal"), and that caveat is the right one — worth restating explicitly here
because it's exactly the kind of *label-construction* leakage this internship's Week-3/Week-6 leakage
hunts are built to catch: does the validation design carry the claim, or does it just confirm the
label's own formula? Here, it mostly does the latter.

**Finding — "What Predicts Growth?" (ML Appendix, logistic regression, 71% holdout accuracy).** The
paper reports content age, days since update, and days visible as the strongest signals separating
growing from declining pages. *My questions:* (1) the paper doesn't state whether the 80/20 holdout
split is row-level or grouped by brand — with 57 brands in the portfolio, a row-level split risks the
same memorization gap this project's own client-grouped split was built to avoid (Section 2 below
re-runs my own model both ways to show the size of that gap). (2) it isn't stated whether "growing"
vs "declining" is measured over a *future* window relative to the features, or over the *same* window
the features come from — if the label window overlaps the feature window, this is a same-window
correlation, not a genuine forecast, which is the exact distinction Section 2 of this internship's
data contract work (Week 3) was built around. Constructive framing: both are answerable with the
methodology detail the paper doesn't publish, not necessarily flaws in the underlying work.


In [1]:
import duckdb, pandas as pd, numpy as np, os, warnings
warnings.filterwarnings("ignore")

candidates = [
    os.path.expanduser("~/Documents/flyrank-hf-data"),
    os.path.expanduser("~/mnt/Documents/flyrank-hf-data"),
]
BASE = next((p for p in candidates if os.path.isdir(p)), candidates[0])
REPO_ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "work")) \
    else os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

con = duckdb.connect()
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
DIM_CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"

feat = con.sql(f"""
    WITH avail AS (SELECT * FROM {FACT} WHERE gsc_data_available IS TRUE),
    prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior, SUM(gsc_clicks) AS clicks_prior,
               AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS avg_position_prior,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impr_prior
        FROM avail WHERE report_date <= DATE '2026-03-15' GROUP BY 1, 2
    ),
    target AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_target, SUM(gsc_clicks) AS clicks_target
        FROM avail WHERE report_date >= DATE '2026-03-16' GROUP BY 1, 2
    )
    SELECT p.*, COALESCE(t.impressions_target, 0) AS impressions_target,
           COALESCE(t.clicks_target, 0) AS clicks_target
    FROM prior p LEFT JOIN target t USING (client_hash_id, content_hash_id)
    WHERE p.impressions_prior >= 10
""").df()

content = con.sql(f"SELECT content_hash_id, content_created_date, content_type FROM {DIM_CONTENT}").df()
feat = feat.merge(content, on="content_hash_id", how="left")
feat["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat["content_created_date"])).dt.days
feat = feat[feat["content_age_days"] >= 0].copy()

feat["imp_rate_prior"]  = feat["impressions_prior"]  / 15.0
feat["imp_rate_target"] = feat["impressions_target"] / 16.0
feat["is_declining"] = (feat["imp_rate_target"] < 0.8 * feat["imp_rate_prior"]).astype(int)
feat["ctr_prior"] = feat["clicks_prior"] / feat["impressions_prior"]

FEATURE_COLS = ["impressions_prior", "clicks_prior", "ctr_prior", "avg_position_prior",
                "days_with_impr_prior", "content_age_days"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

def bucket_pos(p):
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    if p <= 50: return "21-50"
    return "50+"
feat["pos_bucket"] = feat["avg_position_prior"].apply(bucket_pos)

print(f"{len(feat):,} pages / {feat['client_hash_id'].nunique()} clients / decline rate {feat['is_declining'].mean():.3f}")

# Citing the exact figures from the paper being audited above (docs/flyrank-seo-research-march-2026.pdf),
# so the questions in the markdown cell trace back to specific, checkable numbers.
health_score_formula = "Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll depth (20 pts)"
health_rf_top_features = {"Average Position": 0.43, "Impressions": 0.32, "Scroll Depth": 0.15, "CTR": 0.08}
growth_logit_holdout_accuracy = 0.71
growth_logit_split_detail_published = False  # not stated whether row-level or grouped-by-brand

print("Health Score formula:", health_score_formula)
print("RF importance for predicting Health Score:", health_rf_top_features)
print("-> Position + Impressions alone already account for", 
      round(health_rf_top_features["Average Position"] + health_rf_top_features["Impressions"], 2),
      "of total importance - and both are direct inputs to the label being predicted.")
print("\nGrowth model holdout accuracy:", growth_logit_holdout_accuracy)
print("Split granularity (brand-grouped vs row-level) published in the paper:", growth_logit_split_detail_published)


114,715 pages / 39 clients / decline rate 0.344
Health Score formula: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll depth (20 pts)
RF importance for predicting Health Score: {'Average Position': 0.43, 'Impressions': 0.32, 'Scroll Depth': 0.15, 'CTR': 0.08}
-> Position + Impressions alone already account for 0.75 of total importance - and both are direct inputs to the label being predicted.

Growth model holdout accuracy: 0.71
Split granularity (brand-grouped vs row-level) published in the paper: False


## 2. My model under an honest split (before/after)

Re-running my own Week-5 model (`w05_model.ipynb`) on the identical features and label, comparing the
client-grouped split against a random row-level split — the exact question I raised about the paper's
own growth model in Section 1, applied to my own work first.


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score

feature_cols_all = FEATURE_COLS + ["content_type"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(gss.split(feat, groups=feat["client_hash_id"]))
Xg = pd.get_dummies(feat[feature_cols_all], columns=["content_type"])
Xg_tr, Xg_te = Xg.iloc[tr_idx], Xg.iloc[te_idx]
yg_tr, yg_te = feat["is_declining"].iloc[tr_idx], feat["is_declining"].iloc[te_idx]
rf_grouped = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1).fit(Xg_tr, yg_tr)
auc_grouped = roc_auc_score(yg_te, rf_grouped.predict_proba(Xg_te)[:,1])

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xg, feat["is_declining"], test_size=0.2, random_state=42, stratify=feat["is_declining"])
rf_random = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1).fit(Xr_tr, yr_tr)
auc_random = roc_auc_score(yr_te, rf_random.predict_proba(Xr_te)[:,1])

print(f"BEFORE (random row-level 80/20 split, clients leak across train/test): ROC-AUC = {auc_random:.3f}")
print(f"AFTER  (client-grouped 80/20 split, honest):                           ROC-AUC = {auc_grouped:.3f}")
print(f"\nGap: {auc_random - auc_grouped:.3f} ROC-AUC points come from the model partly recognizing")
print("clients it has already seen, rather than a pattern that transfers to a new client. This is the")
print("size of the risk I flagged about the paper's own (unpublished) split granularity in Section 1.")


BEFORE (random row-level 80/20 split, clients leak across train/test): ROC-AUC = 0.678
AFTER  (client-grouped 80/20 split, honest):                           ROC-AUC = 0.586

Gap: 0.092 ROC-AUC points come from the model partly recognizing
clients it has already seen, rather than a pattern that transfers to a new client. This is the
size of the risk I flagged about the paper's own (unpublished) split granularity in Section 1.


## 3. Leakage audit

The same hunt from Week 3 (`w03_data_contract.ipynb`), re-run here on the final feature set actually
used in Weeks 5-8 of this project — reproducing the exact bug I caught and fixed before ever
reporting a capstone number.


In [3]:
# The leakage trap: an earlier version of this project's label used CLICKS instead of impressions,
# with a strict "less than" comparison against the prior window.
feat["is_declining_buggy"] = (feat["clicks_target"] < feat["clicks_prior"]).astype(int)
zero_click_share = (feat["clicks_prior"] == 0).mean()
print(f"Share of pages with clicks_prior == 0: {zero_click_share:.3f}")
print("Those pages can never see clicks_target go negative, so they are MECHANICALLY guaranteed a")
print("'not declining' label under the buggy definition - a spurious deterministic link between")
print("clicks_prior and the label itself, independent of anything the model or the data actually knows.")

Xb = pd.get_dummies(feat[FEATURE_COLS + ["content_type"]], columns=["content_type"])
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
trb, teb = next(gss2.split(feat, groups=feat["client_hash_id"]))
rf_buggy = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1) \
    .fit(Xb.iloc[trb], feat["is_declining_buggy"].iloc[trb])
auc_buggy_model = roc_auc_score(feat["is_declining_buggy"].iloc[teb], rf_buggy.predict_proba(Xb.iloc[teb])[:,1])

test_b = feat.iloc[teb].copy()
expected_ctr_b = feat.iloc[trb].groupby("pos_bucket")["ctr_prior"].mean()
test_b["baseline_score"] = ((test_b["pos_bucket"].map(expected_ctr_b) - test_b["ctr_prior"]).clip(lower=0)) * test_b["impressions_prior"]
auc_buggy_baseline = roc_auc_score(test_b["is_declining_buggy"], test_b["baseline_score"])

print(f"\nBUGGY label -> model ROC-AUC = {auc_buggy_model:.3f}  |  baseline ROC-AUC = {auc_buggy_baseline:.3f}")
print("A model scoring near-perfect next to a baseline scoring WORSE than random is the leakage")
print("signature: 'too good' and 'too bad' next to each other on the same split. That mismatch, not")
print("the high score alone, is what should make anyone distrust a result before reporting it.")
print(f"\nFinal feature set actually used from Week 5 onward: {FEATURE_COLS} - clicks_target and any")
print("target-window field are absent by construction; the corrected label uses an impressions RATE")
print("with a genuine 20%-drop threshold (see w05_model.ipynb), removing the artifact above.")


Share of pages with clicks_prior == 0: 0.577
Those pages can never see clicks_target go negative, so they are MECHANICALLY guaranteed a
'not declining' label under the buggy definition - a spurious deterministic link between
clicks_prior and the label itself, independent of anything the model or the data actually knows.



BUGGY label -> model ROC-AUC = 0.939  |  baseline ROC-AUC = 0.152
A model scoring near-perfect next to a baseline scoring WORSE than random is the leakage
signature: 'too good' and 'too bad' next to each other on the same split. That mismatch, not
the high score alone, is what should make anyone distrust a result before reporting it.

Final feature set actually used from Week 5 onward: ['impressions_prior', 'clicks_prior', 'ctr_prior', 'avg_position_prior', 'days_with_impr_prior', 'content_age_days'] - clicks_target and any
target-window field are absent by construction; the corrected label uses an impressions RATE
with a genuine 20%-drop threshold (see w05_model.ipynb), removing the artifact above.


## 4. Claim rewrite

**Boldest sentence I wrote earlier in this project** (from the Week-4 baseline / capstone draft):
"The model finds pages that are declining."

**Rewritten in safe language:** *Observed* — on the March 2026 partition, pages the random-forest
model ranks highest scored a Precision@20 of 0.75 and Precision@50 of 0.78 on clients held out of
training, versus 0.35 / 0.46 for a position-based heuristic and 0.51 for a random ranking.
*Directional* — pages ranked highest by this model are, on this sample, more likely to see a
meaningful drop in search demand over the next two weeks than pages picked by the heuristic or by no
ranking at all. *Decision-support, not causal or universal* — this ranks a weekly review queue for a
human editor; it does not identify "the" declining pages with certainty, has only been checked on one
month and 8 held-out clients, and makes no claim about *why* a page is declining or what fixing it
would do (no experiment was run to test that).


In [4]:
claim_before = "The model finds pages that are declining."
claim_after = (
    "Observed: Precision@20 0.75 / Precision@50 0.78 (model) vs 0.35/0.46 (CTR-gap heuristic) vs "
    "0.51 (no-skill floor) on 8 held-out clients, March 2026 partition. Directional: highest-ranked "
    "pages are more likely to see a meaningful demand drop in the next two weeks than a heuristic or "
    "no ranking. Decision-support only: this prioritizes a human-reviewed queue, is not causal, and "
    "has not been checked beyond one month / 8 clients."
)
print("BEFORE:", claim_before)
print("\nAFTER: ", claim_after)


BEFORE: The model finds pages that are declining.

AFTER:  Observed: Precision@20 0.75 / Precision@50 0.78 (model) vs 0.35/0.46 (CTR-gap heuristic) vs 0.51 (no-skill floor) on 8 held-out clients, March 2026 partition. Directional: highest-ranked pages are more likely to see a meaningful demand drop in the next two weeks than a heuristic or no ranking. Decision-support only: this prioritizes a human-reviewed queue, is not causal, and has not been checked beyond one month / 8 clients.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
